# 🏠 주택 가격 예측 — 팀 베이스라인
- 데이터: 과제에서 받은 `train.csv`(정답 있음), `test.csv`(정답 없음 → 제출용)
- 예측할 값: `SalePrice`
- **평가지표: RMSE** (낮을수록 좋음)
- 비교 기준: 과제 기본 노트북(LinearRegression)의 Validation RMSE **≈ 39,487**

사용법
1. 위에서부터 순서대로 실행 (Shift + Enter)
2. 점수 올리고 싶으면 **3번 셀**(피처 만들기)과 **4번 셀**(이상치 제거)을 손보면서 다시 실행
3. 마지막 셀이 `submission_baseline.csv`를 새로 만들어 줌 (기존 `submission.csv`는 그대로 둠)

⚠️ 팀원끼리 점수 비교하려면 `RANDOM_STATE`, `TEST_SIZE`는 바꾸지 말기 (검증 데이터가 같아야 공정함)

In [ ]:
# 1. 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import root_mean_squared_error
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

RANDOM_STATE = 42   # 과제 기본 노트북과 같은 값 → 같은 검증 데이터로 비교 가능
TEST_SIZE    = 0.2
USE_LOG      = True # 집값에 log 씌워서 학습할지 (보통 True가 RMSE 더 낮음)

# 그래프 한글 깨짐 방지 (윈도우: 맑은 고딕 / 맥: 애플고딕)
import platform
plt.rcParams['font.family'] = 'AppleGothic' if platform.system() == 'Darwin' else 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 2. 데이터 불러오기 + 훑어보기
TARGET = 'SalePrice'
train_df = pd.read_csv('./train.csv')
test_df  = pd.read_csv('./test.csv')
print('train:', train_df.shape, ' test:', test_df.shape)

# 결측치 많은 컬럼 TOP 10
print('\n[결측치 개수]')
print(train_df.isnull().sum().sort_values(ascending=False).head(10))

# 타깃(집값) 분포 — 오른쪽으로 꼬리가 길면 log 씌우는 게 효과 좋음
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(train_df[TARGET], bins=50); ax[0].set_title('SalePrice')
ax[1].hist(np.log1p(train_df[TARGET]), bins=50); ax[1].set_title('log(SalePrice)')
plt.show()

In [ ]:
# 3. 데이터 정리 + 새 피처 만들기
#    train.csv와 test.csv에 똑같이 적용해야 해서 함수로 만들어 둠

# 이 데이터에서 NaN은 대부분 '없음'이라는 뜻 (예: PoolQC가 NaN = 수영장 없음)
#  → 최빈값으로 채우면 '없는 수영장'이 '있는 수영장'이 되니까 'None' / 0으로 채움
NONE_COLS = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'MasVnrType',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
ZERO_COLS = ['MasVnrArea', 'GarageCars', 'GarageArea',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath']

def prepare(d):
    d = d.drop(columns=['Id'])
    d[NONE_COLS] = d[NONE_COLS].fillna('None')
    d[ZERO_COLS] = d[ZERO_COLS].fillna(0)
    d['MSSubClass'] = d['MSSubClass'].astype(str)   # 숫자처럼 보이지만 '주택 유형 코드'라서 문자로

    # (선택) 새 피처 — 주석(#) 풀어서 하나씩 실험해보기
    # d['TotalSF']   = d['TotalBsmtSF'] + d['1stFlrSF'] + d['2ndFlrSF']   # 총 면적
    # d['HouseAge']  = d['YrSold'] - d['YearBuilt']                        # 집 나이
    # d['RemodAge']  = d['YrSold'] - d['YearRemodAdd']                     # 리모델링 후 경과 연수
    # d['TotalBath'] = d['FullBath'] + 0.5*d['HalfBath'] + d['BsmtFullBath'] + 0.5*d['BsmtHalfBath']
    return d

data   = prepare(train_df)
test_X = prepare(test_df)
print(data.shape, test_X.shape)

In [ ]:
# 4. X(문제), y(정답) 나누고 → train / validation 나누기
X = data.drop(columns=[TARGET])
y = data[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# (선택) 이상치 제거 — ⚠️ 반드시 split 후 train에만! (validation까지 지우면 점수가 부풀려지고 팀원끼리 비교 불가)
# 유명한 이상치: 면적은 엄청 큰데 가격은 싼 집
# out = (X_train['GrLivArea'] > 4000) & (y_train < 300000)
# X_train, y_train = X_train[~out], y_train[~out]

# 숫자형 / 문자형 컬럼 자동으로 나누기
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
print('숫자형:', len(num_cols), '개 / 문자형:', len(cat_cols), '개')
print('train:', X_train.shape, ' validation:', X_val.shape)

In [ ]:
# 5. 전처리 규칙 만들기 (결측치 채우기 + 문자형→숫자 변환 + 스케일링)
#    → 모델마다 자동으로 붙여서 쓰니까 따로 신경 안 써도 됨
#    (train으로만 규칙을 배우고 validation/test에는 적용만 해서 데이터 누수가 없음)
preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),         # 숫자 결측치 → 중앙값
        ('scale',  StandardScaler()),                         # 크기 맞추기 (회귀용)
    ]), num_cols),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),  # 남은 문자 결측치 → 최빈값
        ('onehot', OneHotEncoder(handle_unknown='ignore',     # test에만 있는 값은 무시
                                 min_frequency=10,
                                 sparse_output=False)),       # 문자 → 0/1 컬럼
    ]), cat_cols),
])

def make_model(regressor):
    """전처리 + 모델 + (선택) log 변환을 한 번에 묶어주는 함수"""
    pipe = Pipeline([('prep', preprocess), ('model', regressor)])
    if USE_LOG:
        # log로 학습 → 예측할 땐 자동으로 원래 집값으로 되돌림 (RMSE는 원래 단위로 계산됨)
        return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
    return pipe

In [ ]:
# 6. 비교할 모델들
models = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=10),
    'DecisionTree':     DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    'RandomForest':     RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE),
    'GradientBoosting': HistGradientBoostingRegressor(learning_rate=0.05, max_iter=500,
                                                      random_state=RANDOM_STATE),
}

# (선택) LightGBM이 설치돼 있으면 같이 비교 — 없으면 그냥 건너뜀
try:
    from lightgbm import LGBMRegressor
    models['LightGBM'] = LGBMRegressor(n_estimators=1000, learning_rate=0.03,
                                       num_leaves=31, random_state=RANDOM_STATE, verbose=-1)
except ImportError:
    print('LightGBM 없음 → 건너뜀 (설치: pip install lightgbm)')

In [ ]:
# 7. 학습 → RMSE 비교  ★ 대결 점수 ★
#    Val_RMSE : validation 한 번으로 잰 점수 (과제 기본 노트북과 같은 방식)
#    CV_RMSE  : train.csv 전체를 5번 다르게 나눠서 잰 평균 → 운에 덜 흔들림, 1등은 이걸로 뽑음
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results, fitted = [], {}

for name, reg in models.items():
    m = make_model(reg)
    m.fit(X_train, y_train)
    val_rmse = root_mean_squared_error(y_val, m.predict(X_val))
    cv_rmse  = -cross_val_score(make_model(reg), X, y, cv=cv,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    results.append({'model': name, 'Val_RMSE': val_rmse,
                    'CV_RMSE': cv_rmse.mean(), 'CV_std': cv_rmse.std()})
    fitted[name] = m
    print(f'{name:18s}  Val_RMSE = {val_rmse:,.0f}   CV_RMSE = {cv_rmse.mean():,.0f} (± {cv_rmse.std():,.0f})')

result_df = pd.DataFrame(results).sort_values('CV_RMSE').reset_index(drop=True)
best_name = result_df.loc[0, 'model']
print('\n🏆 최고 모델 (CV_RMSE 기준):', best_name)
print('   (참고: 과제 기본 노트북 Val_RMSE ≈ 39,487)')
result_df.round(0)

In [ ]:
# 8. 실제값 vs 예측값 그래프 — 점이 빨간 대각선에 붙어 있을수록 잘 맞춘 것
best = fitted[best_name]
pred = best.predict(X_val)

plt.figure(figsize=(5, 5))
plt.scatter(y_val, pred, alpha=0.4, s=10)
lims = [min(y_val.min(), pred.min()), max(y_val.max(), pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('실제 집값'); plt.ylabel('예측 집값'); plt.title(best_name)
plt.show()

In [ ]:
# 9. 어떤 변수가 집값 예측에 중요했나? (permutation importance)
#    해당 컬럼을 뒤섞었을 때 성능이 많이 떨어질수록 = 중요한 변수
imp = permutation_importance(best, X_val, y_val, n_repeats=5,
                             random_state=RANDOM_STATE, n_jobs=-1)
imp_s = pd.Series(imp.importances_mean, index=X_val.columns).sort_values().tail(15)

plt.figure(figsize=(6, 5))
imp_s.plot.barh()
plt.title(f'중요 변수 TOP 15 ({best_name})')
plt.show()

In [ ]:
# 10. 제출 파일 만들기 — 1등 모델을 train.csv 전체로 다시 학습 → test.csv 예측
final = make_model(models[best_name])
final.fit(X, y)

submission = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': final.predict(test_X)})
SUBMIT_PATH = 'submission_baseline.csv'   # 기존 submission.csv를 덮어쓰지 않도록 다른 이름으로 저장
submission.to_csv(SUBMIT_PATH, index=False)
print('저장 완료:', SUBMIT_PATH, submission.shape)
submission.head()

## 💡 점수 더 올리고 싶을 때 체크리스트
- [ ] `USE_LOG = True` / `False` 둘 다 해보고 RMSE 비교
- [ ] 3번 셀: 새 피처(총 면적, 집 나이 등) 주석 풀어보기 → 9번 그래프에서 중요한 변수 위주로
- [ ] 4번 셀: 이상치 제거 주석 풀어보기 (train에서만 지우기!)
- [ ] 6번 셀: GradientBoosting의 `learning_rate`(0.03~0.1), `max_iter`(300~1500), Ridge의 `alpha` 바꿔보기
- [ ] 바꿀 때마다 **한 번에 하나씩만** 바꾸고 CV_RMSE 기록하기 (뭐가 효과 있었는지 알 수 있게!)